# KYRBS 2023–2024: SAS → R analysis

This notebook reproduces the main steps in the supplied SAS program using R and the `survey` package.

**Flow:** import data → BMI classification → smartphone-use groups → covariates → missing-data exclusion → survey design → Table 1 (moonBook + Excel) → obesity prevalence (Table 2) → time-specific odds ratios.

The local SAS data files are expected in:
`C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data`

## Cell 1 — Install packages (run once)

If these packages are already installed, skip this cell.

In [1]:
install.packages(c("haven", "dplyr", "tidyr", "survey", "broom", "moonBook", "openxlsx"))

Installing packages into 'C:/Users/picks/AppData/Local/R/win-library/4.6'
(as 'lib' is unspecified)



package 'haven' successfully unpacked and MD5 sums checked


Warning message:
"cannot remove prior installation of package 'haven'"
Warning message in file.copy(savedcopy, lib, recursive = TRUE):
"problem copying C:\Users\picks\AppData\Local\R\win-library\4.6\00LOCK\haven\libs\x64\haven.dll to C:\Users\picks\AppData\Local\R\win-library\4.6\haven\libs\x64\haven.dll: Permission denied"
Warning message:
"restored 'haven'"


package 'dplyr' successfully unpacked and MD5 sums checked


Warning message:
"cannot remove prior installation of package 'dplyr'"
Warning message in file.copy(savedcopy, lib, recursive = TRUE):
"problem copying C:\Users\picks\AppData\Local\R\win-library\4.6\00LOCK\dplyr\libs\x64\dplyr.dll to C:\Users\picks\AppData\Local\R\win-library\4.6\dplyr\libs\x64\dplyr.dll: Permission denied"
Warning message:
"restored 'dplyr'"


...

package 'survey' successfully unpacked and MD5 sums checked


Warning message:
"cannot remove prior installation of package 'survey'"
Warning message in file.copy(savedcopy, lib, recursive = TRUE):
"problem copying C:\Users\picks\AppData\Local\R\win-library\4.6\00LOCK\survey\libs\x64\survey.dll to C:\Users\picks\AppData\Local\R\win-library\4.6\survey\libs\x64\survey.dll: Permission denied"
Warning message:
"restored 'survey'"


package 'broom' successfully unpacked and MD5 sums checked
package 'moonBook' successfully unpacked and MD5 sums checked
package 'openxlsx' successfully unpacked and MD5 sums checked


Warning message:
"cannot remove prior installation of package 'openxlsx'"
Warning message in file.copy(savedcopy, lib, recursive = TRUE):
"problem copying C:\Users\picks\AppData\Local\R\win-library\4.6\00LOCK\openxlsx\libs\x64\openxlsx.dll to C:\Users\picks\AppData\Local\R\win-library\4.6\openxlsx\libs\x64\openxlsx.dll: Permission denied"
Warning message:
"restored 'openxlsx'"



The downloaded binary packages are in
	C:\Users\Public\Documents\ESTsoft\CreatorTemp\RtmpemNz9R\downloaded_packages


## Cell 2 — Load packages

In [2]:
library(haven)
library(dplyr)
library(tidyr)
library(survey)
library(broom)
library(moonBook)
library(openxlsx)

Warning message:
"package 'haven' was built under R version 4.6.1"
Warning message:
"package 'dplyr' was built under R version 4.6.1"

Attaching package: 'dplyr'


The following objects are masked from 'package:stats':

    filter, lag


The following objects are masked from 'package:base':

    intersect, setdiff, setequal, union


Warning message:
"package 'tidyr' was built under R version 4.6.1"
Warning message:
"package 'survey' was built under R version 4.6.1"
Loading required package: grid

Loading required package: Matrix


Attaching package: 'Matrix'


The following objects are masked from 'package:tidyr':

    expand, pack, unpack


Loading required package: survival


Attaching package: 'survey'


The following object is masked from 'package:graphics':

    dotchart


Warning message:
"package 'broom' was built under R version 4.6.1"
Warning message:
"package 'moonBook' was built under R version 4.6.1"
Warning message:
"package 'openxlsx' was built under R version 4.6.1"


## Cell 3 — Set the data directory and check files

In [3]:
DATA_DIR <- "C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data"

FILE_2023 <- file.path(DATA_DIR, "kyrbs2023.sas7bdat")
FILE_2024 <- file.path(DATA_DIR, "kyrbs2024.sas7bdat")

cat("2023 exists:", file.exists(FILE_2023), "\n")
cat("2024 exists:", file.exists(FILE_2024), "\n")

if (!file.exists(FILE_2023) || !file.exists(FILE_2024)) {
  stop("SAS files were not found. Check DATA_DIR and filenames.")
}

2023 exists: TRUE 
2024 exists: TRUE 


## Cell 4 — Read 2023 and 2024 SAS datasets and combine them

This corresponds to the SAS `data ky; set a.kyrbs2023 a.kyrbs2024; run;` step.

In [4]:
DATA_DIR <- "C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data"

FILE_2023 <- file.path(DATA_DIR, "kyrbs2023.sas7bdat")
FILE_2024 <- file.path(DATA_DIR, "kyrbs2024.sas7bdat")

FILE_2023
FILE_2024

file.exists(FILE_2023)
file.exists(FILE_2024)

KY23 <- haven::read_sas(FILE_2023, encoding = "CP949")
KY24 <- haven::read_sas(FILE_2024, encoding = "CP949")

KY <- dplyr::bind_rows(KY23, KY24)

dim(KY)

[1] "C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data/kyrbs2023.sas7bdat"

[1] "C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data/kyrbs2024.sas7bdat"

[1] TRUE

[1] TRUE

[1] 107533    210

In [5]:
head(names(KY), 30)
stopifnot("HT" %in% names(KY), "WT" %in% names(KY))

[1] "OBS"        "mod_d"      "YEAR"       "CITY"       "CTYPE"     
 [6] "CTYPE_SD"   "MH"         "SCHOOL"     "STYPE"      "STRATA"    
[11] "STRATA_NM"  "CLUSTER"    "GROUP"      "W"          "PR_HT"     
[16] "PR_BI"      "PR_HD"      "F_BR"       "F_FRUIT"    "F_SWD_A"   
[21] "F_FASTFOOD" "F_EDU"      "F_WAT"      "PA_TOT"     "PA_VIG_D"  
[26] "PA_MSC"     "PA_SWD_S"   "PA_SWD_N"   "PA_SWK_S"   "PA_SWK_N"

## Cell 5 — Calculate BMI

In [6]:
A2 <- KY %>%
  mutate(
    K = HT / 100,
    BMI = round(WT / (K * K), 9)
  )

## Cell 6 — Assign sex/age-specific BMI percentile cutoffs

In [7]:
A2 <- A2 %>%
  mutate(
    PCT05 = case_when(
      SEX == 1 & AGE_M == 144 ~ 15.5,
      SEX == 1 & AGE_M == 145 ~ 15.6,
      SEX == 1 & AGE_M == 146 ~ 15.6,
      SEX == 1 & AGE_M >= 147 & AGE_M <= 227 ~ 17.0,
      SEX == 2 & AGE_M == 144 ~ 15.3,
      SEX == 2 & AGE_M == 145 ~ 15.3,
      SEX == 2 & AGE_M == 146 ~ 15.4,
      SEX == 2 & AGE_M >= 147 & AGE_M <= 227 ~ 16.5,
      TRUE ~ NA_real_
    ),
    PCT85 = case_when(
      SEX == 1 & AGE_M == 144 ~ 23.0,
      SEX == 1 & AGE_M == 145 ~ 23.0,
      SEX == 1 & AGE_M == 146 ~ 23.1,
      SEX == 1 & AGE_M >= 147 & AGE_M <= 227 ~ 24.5,
      SEX == 2 & AGE_M == 144 ~ 22.1,
      SEX == 2 & AGE_M == 145 ~ 22.2,
      SEX == 2 & AGE_M == 146 ~ 22.2,
      SEX == 2 & AGE_M >= 147 & AGE_M <= 227 ~ 23.5,
      TRUE ~ NA_real_
    ),
    PCT95 = case_when(
      SEX == 1 & AGE_M == 144 ~ 25.1,
      SEX == 1 & AGE_M == 145 ~ 25.1,
      SEX == 1 & AGE_M == 146 ~ 25.2,
      SEX == 1 & AGE_M >= 147 & AGE_M <= 227 ~ 26.5,
      SEX == 2 & AGE_M == 144 ~ 24.1,
      SEX == 2 & AGE_M == 145 ~ 24.2,
      SEX == 2 & AGE_M == 146 ~ 24.2,
      SEX == 2 & AGE_M >= 147 & AGE_M <= 227 ~ 25.5,
      TRUE ~ NA_real_
    )
  )

## Cell 7 — Create BMI groups (`g_bmi`)

In [8]:
A3 <- A2 %>%
  mutate(
    G_BMI = case_when(
      !is.na(BMI) & !is.na(PCT05) & !is.na(PCT85) & !is.na(PCT95) & BMI >= PCT95 ~ 4,
      !is.na(BMI) & !is.na(PCT05) & !is.na(PCT85) & !is.na(PCT95) & BMI >= PCT85 & BMI < PCT95 ~ 3,
      !is.na(BMI) & !is.na(PCT05) & !is.na(PCT85) & !is.na(PCT95) & BMI >= PCT05 & BMI < PCT85 ~ 2,
      !is.na(BMI) & !is.na(PCT05) & !is.na(PCT85) & !is.na(PCT95) & BMI < PCT05 ~ 1,
      TRUE ~ NA_real_
    )
  )

## Cell 8 — Calculate average daily smartphone use

In [9]:
A4 <- A3 %>%
  mutate(
    SP_WD_HR = INT_SPWD_TM / 60,
    SP_WK_HR = INT_SPWK_TM / 60,
    SP_AVG = (SP_WD_HR * 5 + SP_WK_HR * 2) / 7
  )

## Cell 9 — Create smartphone-use groups (`time`)

In [10]:
A4 <- A4 %>%
  mutate(
    TIME = case_when(
      SP_AVG < 2 ~ 1,
      SP_AVG >= 2 & SP_AVG < 4 ~ 2,
      SP_AVG >= 4 & SP_AVG < 6 ~ 3,
      SP_AVG >= 6 ~ 4,
      TRUE ~ NA_real_
    )
  )

## Cell 10 — Create region, smoking, education, economic status, stress, depression, drinking, and age group

In [11]:
A4 <- A4 %>%
  mutate(
    REGION = case_when(
      as.character(CTYPE) == "군지역" ~ 2,
      as.character(CTYPE) %in% c("대도시", "중소도시") ~ 1,
      TRUE ~ NA_real_
    ),

    SMOKING1 = if_else(TC_DAYS %in% c(1, 9999) | is.na(TC_DAYS), 0, 1),
    SMOKING2 = if_else(TC_EC_MN %in% c(1, 9999) | is.na(TC_EC_MN), 0, 1),
    SMOKING3 = if_else(TC_HTP_MN %in% c(1, 9999) | is.na(TC_HTP_MN), 0, 1),
    SMOKING = if_else(SMOKING1 == 1 | SMOKING2 == 1 | SMOKING3 == 1, 1, 0),

    EDU = case_when(
      E_S_RCRD %in% c(1, 2) ~ 1,
      E_S_RCRD == 3 ~ 2,
      E_S_RCRD %in% c(4, 5) ~ 3,
      TRUE ~ NA_real_
    ),

    ECO = case_when(
      E_SES %in% c(1, 2) ~ 1,
      E_SES == 3 ~ 2,
      E_SES %in% c(4, 5) ~ 3,
      TRUE ~ NA_real_
    ),

    STRESS = if_else(M_STR %in% c(1, 2), 1, 0),
    DEPRESS = if_else(M_SAD == 2, 1, 0),
    DRINKING = if_else(AC_DAYS %in% c(1, 9999) | is.na(AC_DAYS), 0, 1),

    AGE_G = case_when(
      AGE %in% c(12, 13, 14) ~ 1,
      AGE %in% c(15, 16, 17, 18) ~ 2,
      TRUE ~ NA_real_
    )
  ) %>%
  filter(YEAR %in% c(2023, 2024))

## Cell 11 — Keep the same analysis variables as the SAS program

In [12]:
A4 <- A4 %>%
  select(
    YEAR, AGE_G, SEX, REGION, G_BMI, TIME,
    SMOKING, DRINKING, EDU, ECO, STRESS, DEPRESS,
    W, CLUSTER, STRATA
  )

## Cell 12 — Check frequencies before deleting missing observations

In [13]:
freq_vars <- c("YEAR", "SEX", "AGE_G", "REGION", "G_BMI", "EDU", "ECO",
               "SMOKING", "DRINKING", "TIME", "STRESS", "DEPRESS")

for (v in freq_vars) {
  cat("\n====================", v, "====================\n")
  print(table(A4[[v]], useNA = "ifany"))
}


==================== YEAR ====================

 2023  2024 
52880 54653 

==================== SEX ====================

    1     2 
54859 52674 

==================== AGE_G ====================

    1     2  <NA> 
45619 61787   127 

==================== REGION ====================

    1     2 
99520  8013 

==================== G_BMI ====================

    1     2     3     4  <NA> 
 7102 74988 10028 12522  2893 

==================== EDU ====================

    1     2     3  <NA> 
40879 31384 35262     8 

==================== ECO ====================

    1     2     3  <NA> 
45553 49412 12558    10 

==================== SMOKING ====================

     0      1 
102406   5127 

==================== DRINKING ====================

    0     1 
96504 11029 

==================== TIME ====================

    1     2     3     4  <NA> 
 7598 32989 31714 31129  4103 

==================== STRESS ====================

    0     1 
64792 42741 

==================== DEPRESS

## Cell 13 — Exclude records with missing age group, BMI group, education, or economic status

In [14]:
A5 <- A4 %>%
  filter(
    !is.na(AGE_G),
    !is.na(G_BMI),
    !is.na(EDU),
    !is.na(ECO)
  )

## Cell 14 — Create the binary obesity outcome

In [15]:
A5 <- A5 %>%
  mutate(
    OBESE = case_when(
      G_BMI %in% c(1, 2, 3) ~ 0,
      G_BMI == 4 ~ 1,
      TRUE ~ NA_real_
    )
  )

## Cell 15 — Check the final analytic sample

In [16]:
cat("Final N:", nrow(A5), "\n")

for (v in c(freq_vars, "OBESE")) {
  cat("\n====================", v, "====================\n")
  print(table(A5[[v]], useNA = "ifany"))
}

Final N: 104630 

==================== YEAR ====================

 2023  2024 
51462 53168 

==================== SEX ====================

    1     2 
53469 51161 

==================== AGE_G ====================

    1     2 
44499 60131 

==================== REGION ====================

    1     2 
96912  7718 

==================== G_BMI ====================

    1     2     3     4 
 7102 74981 10027 12520 

==================== EDU ====================

    1     2     3 
40050 30681 33899 

==================== ECO ====================

    1     2     3 
44496 48177 11957 

==================== SMOKING ====================

    0     1 
99861  4769 

==================== DRINKING ====================

    0     1 
94107 10523 

==================== TIME ====================

    1     2     3     4  <NA> 
 7461 32421 31025 30013  3710 

==================== STRESS ====================

    0     1 
63335 41295 

==================== DEPRESS ====================

    0     1 

## Cell 16 — Define the complex survey design

In [17]:
options(survey.lonely.psu = "adjust")

DESIGN <- svydesign(
  ids = ~CLUSTER,
  STRATA = ~STRATA,
  weights = ~W,
  data = A5,
  nest = TRUE
)

DESIGN

1 - level Cluster Sampling design (with replacement)
With (800) clusters.
svydesign(ids = ~CLUSTER, STRATA = ~STRATA, weights = ~W, data = A5, 
    nest = TRUE)

## Cell 17 — Table 1: create the table with `moonBook`

In [18]:
A5_labeled <- A5 %>%
  mutate(
    TIME = factor(TIME, levels = 1:4, labels = c("<2h", "2h–4h", "4h–6h", ">6h")),
    SEX = factor(SEX, levels = c(1, 2), labels = c("Male", "Female")),
    AGE_G = factor(AGE_G, levels = c(1, 2), labels = c("7th–9th grade", "10th–12th grade")),
    REGION = factor(REGION, levels = c(1, 2), labels = c("Urban", "Rural")),
    G_BMI = factor(G_BMI, levels = c(1, 2, 3, 4),
                   labels = c("Underweight", "Normal", "Overweight", "Obese")),
    EDU = factor(EDU, levels = c(1, 2, 3), labels = c("High", "Middle", "Low")),
    ECO = factor(ECO, levels = c(1, 2, 3), labels = c("High", "Middle", "Low")),
    STRESS = factor(STRESS, levels = c(0, 1), labels = c("Low", "High")),
    DEPRESS = factor(DEPRESS, levels = c(0, 1), labels = c("Low", "High")),
    SMOKING = factor(SMOKING, levels = c(0, 1), labels = c("Non-smoker", "Smoker")),
    DRINKING = factor(DRINKING, levels = c(0, 1), labels = c("Non-drinker", "Drinker"))
  )

moonbook_table1 <- mytable(
  TIME ~ SEX + AGE_G + REGION + G_BMI + EDU + ECO + STRESS + DEPRESS +
    SMOKING + DRINKING,
  data = A5_labeled,
  show.total = TRUE,
  show.all = FALSE
)

moonbook_table1


                               Descriptive Statistics by 'TIME'                               
——————————————————————————————————————————————————————————————————————————————————————————————— 
                         <2h          2h–4h         4h–6h          >6h          Total       p  
                       (N=7461)     (N=32421)     (N=31025)     (N=30013)    (N=100920)  
——————————————————————————————————————————————————————————————————————————————————————————————— 
 SEX                                                                                      0.000
   - Male            4658 (62.4%) 18285 (56.4%) 15020 (48.4%) 12921 (43.1%) 50884 (50.4%)      
   - Female          2803 (37.6%) 14136 (43.6%) 16005 (51.6%) 17092 (56.9%) 50036 (49.6%)      
 AGE_G                                                                                    0.000
   - 7th–9th grade   4285 (57.4%) 15372 (47.4%) 12432 (40.1%) 11239 (37.4%) 43328 (42.9%)      
   - 10th–12th grade 3176 (42.6%) 17049 (52.

## Cell 18 — Table 1: make the manuscript-style Excel table

In [19]:
TABLE1_VARS <- c(
  SEX = "Sex",
  AGE_G = "Grade",
  REGION = "Region of residence",
  G_BMI = "BMI*",
  EDU = "Academic achievement",
  ECO = "Economic level",
  STRESS = "Stress",
  DEPRESS = "Depression",
  SMOKING = "Smoking status",
  DRINKING = "Alcohol consumption"
)

TIME_LEVELS <- c("<2h", "2h–4h", "4h–6h", ">6h")

make_table1 <- function(data, variables, time_var = "TIME") {
  data[[time_var]] <- as.character(data[[time_var]])
  total_n <- nrow(data)
  result <- list()
  k <- 1

  # Overall row
  overall_row <- data.frame(
    Characteristics = "Overall",
    Total = sprintf("%d", total_n),
    `<2h` = sprintf("%d (%.2f)", sum(!is.na(data[[time_var]]) & data[[time_var]] == "<2h"),
                    mean(!is.na(data[[time_var]]) & data[[time_var]] == "<2h") * 100),
    `2h–4h` = sprintf("%d (%.2f)", sum(!is.na(data[[time_var]]) & data[[time_var]] == "2h–4h"),
                      mean(!is.na(data[[time_var]]) & data[[time_var]] == "2h–4h") * 100),
    `4h–6h` = sprintf("%d (%.2f)", sum(!is.na(data[[time_var]]) & data[[time_var]] == "4h–6h"),
                      mean(!is.na(data[[time_var]]) & data[[time_var]] == "4h–6h") * 100),
    `>6h` = sprintf("%d (%.2f)", sum(!is.na(data[[time_var]]) & data[[time_var]] == ">6h"),
                    mean(!is.na(data[[time_var]]) & data[[time_var]] == ">6h") * 100),
    check.names = FALSE,
    stringsAsFactors = FALSE
  )
  result[[k]] <- overall_row
  k <- k + 1

  for (v in names(variables)) {
    # Section/header row
    result[[k]] <- data.frame(
      Characteristics = variables[[v]],
      Total = "",
      `<2h` = "",
      `2h–4h` = "",
      `4h–6h` = "",
      `>6h` = "",
      check.names = FALSE,
      stringsAsFactors = FALSE
    )
    k <- k + 1

    levs <- levels(factor(data[[v]]))

    for (lev in levs) {
      total_count <- sum(!is.na(data[[v]]) & data[[v]] == lev)
      total_pct <- ifelse(total_n > 0, total_count / total_n * 100, NA_real_)

      row <- data.frame(
        Characteristics = paste0("  ", lev),
        Total = sprintf("%d (%.2f)", total_count, total_pct),
        `<2h` = "",
        `2h–4h` = "",
        `4h–6h` = "",
        `>6h` = "",
        check.names = FALSE,
        stringsAsFactors = FALSE
      )

      for (tt in TIME_LEVELS) {
        d_group <- data[!is.na(data[[time_var]]) & data[[time_var]] == tt & !is.na(data[[v]]), , drop = FALSE]
        n_group <- nrow(d_group)
        n_level <- sum(d_group[[v]] == lev)
        pct_group <- ifelse(n_group > 0, n_level / n_group * 100, NA_real_)
        row[[tt]] <- sprintf("%d (%.2f)", n_level, pct_group)
      }

      result[[k]] <- row
      k <- k + 1
    }
  }

  bind_rows(result)
}

Table1 <- make_table1(A5_labeled, TABLE1_VARS)
Table1

Characteristics,Total,<2h,2h–4h,4h–6h,>6h
<chr>,<chr>,<chr>,<chr>,<chr>,<chr>
Overall,104630,7461 (7.13),32421 (30.99),31025 (29.65),30013 (28.68)
Sex,,,,,
Male,53469 (51.10),4658 (62.43),18285 (56.40),15020 (48.41),12921 (43.05)
Female,51161 (48.90),2803 (37.57),14136 (43.60),16005 (51.59),17092 (56.95)
Grade,,,,,
7th–9th grade,44499 (42.53),4285 (57.43),15372 (47.41),12432 (40.07),11239 (37.45)
10th–12th grade,60131 (57.47),3176 (42.57),17049 (52.59),18593 (59.93),18774 (62.55)
Region of residence,,,,,
Urban,96912 (92.62),7053 (94.53),30379 (93.70),28648 (92.34),27406 (91.31)


## Cell 18 Export — Export Table 1 to Excel

In [20]:
TABLE1_XLSX <- file.path(DATA_DIR, "Table1_screen_time.xlsx")

wb <- createWorkbook()
addWorksheet(wb, "Table 1")

writeData(wb, "Table 1", Table1, startRow = 1, startCol = 1, rowNames = FALSE)

header_style <- createStyle(textDecoration = "bold", halign = "center", border = "Bottom")
section_style <- createStyle(textDecoration = "bold")

addStyle(wb, "Table 1", header_style,
         rows = 1, cols = 1:ncol(Table1), gridExpand = TRUE)

section_rows <- which(Table1$Total == "" & Table1$Characteristics != "Overall") + 1
if (length(section_rows) > 0) {
  addStyle(wb, "Table 1", section_style,
           rows = section_rows, cols = 1, gridExpand = FALSE)
}

setColWidths(wb, "Table 1", cols = 1, widths = 30)
setColWidths(wb, "Table 1", cols = 2:ncol(Table1), widths = 18)
freezePane(wb, "Table 1", firstRow = TRUE)

saveWorkbook(wb, TABLE1_XLSX, overwrite = TRUE)

## Cell 19 — Table 2: weighted obesity prevalence by screen time

This section reproduces manuscript Table 2 using the complex survey design. Values are weighted obesity prevalence (%) with 95% confidence intervals (CIs), stratified by daily screen-time group.


In [21]:
TABLE2_VARS <- c(
  SEX = "Sex",
  AGE_G = "Grade",
  REGION = "Region of residence",
  EDU = "Education level",
  ECO = "Economic level",
  DEPRESS = "Depression",
  STRESS = "Stress",
  SMOKING = "Smoking status",
  DRINKING = "Alcohol consumption"
)

TIME_LEVELS <- c("<2h", "2h–4h", "4h–6h", ">6h")

A5_T2 <- A5 %>%
  mutate(
    TIME = factor(TIME, levels = 1:4, labels = TIME_LEVELS),
    SEX = factor(SEX, levels = c(1, 2), labels = c("Male", "Female")),
    AGE_G = factor(AGE_G, levels = c(1, 2),
                   labels = c("7th–9th grade (middle school)",
                              "10th–12th grade (high school)")),
    REGION = factor(REGION, levels = c(1, 2), labels = c("Urban", "Rural")),
    EDU = factor(EDU, levels = c(1, 2, 3), labels = c("High", "Middle", "Low")),
    ECO = factor(ECO, levels = c(1, 2, 3), labels = c("High", "Middle", "Low")),
    DEPRESS = factor(DEPRESS, levels = c(0, 1), labels = c("Low", "High")),
    STRESS = factor(STRESS, levels = c(0, 1), labels = c("Low", "High")),
    SMOKING = factor(SMOKING, levels = c(0, 1), labels = c("Non-smoker", "Smoker")),
    DRINKING = factor(DRINKING, levels = c(0, 1),
                       labels = c("Non-drinker", "Drinker"))
  )

TABLE2_DESIGN <- svydesign(
  ids = ~CLUSTER,
  strata = ~STRATA,
  weights = ~W,
  data = A5_T2,
  nest = TRUE
)

TABLE2_LEVELS <- list(
  SEX = c("Male", "Female"),
  AGE_G = c("7th–9th grade (middle school)",
            "10th–12th grade (high school)"),
  REGION = c("Urban", "Rural"),
  EDU = c("High", "Middle", "Low"),
  ECO = c("High", "Middle", "Low"),
  DEPRESS = c("Low", "High"),
  STRESS = c("Low", "High"),
  SMOKING = c("Non-smoker", "Smoker"),
  DRINKING = c("Non-drinker", "Drinker")
)


In [22]:
weighted_prev <- function(design, subgroup_var = NULL, subgroup_level = NULL) {

  if (!is.null(subgroup_var)) {
    keep <- !is.na(design$variables[[subgroup_var]]) &
      design$variables[[subgroup_var]] == subgroup_level
    design <- design[keep, ]
  }

  results <- lapply(TIME_LEVELS, function(tt) {
    d_time <- subset(design, TIME == tt)

    if (nrow(d_time) == 0) {
      return(data.frame(
        TIME = tt,
        prevalence = NA_real_,
        lower = NA_real_,
        upper = NA_real_,
        stringsAsFactors = FALSE
      ))
    }

    est <- svymean(~OBESE, d_time, na.rm = TRUE)

    # Direct 95% CI from estimate and standard error.
    # This avoids both svyby() and confint.svyby() issues.
    estimate <- as.numeric(coef(est)[1])
    se <- as.numeric(SE(est)[1])
    z <- qnorm(0.975)

    lower <- max(0, estimate - z * se)
    upper <- min(1, estimate + z * se)

    data.frame(
      TIME = tt,
      prevalence = estimate * 100,
      lower = lower * 100,
      upper = upper * 100,
      stringsAsFactors = FALSE
    )
  })

  bind_rows(results)
}

format_prev <- function(x) {
  ifelse(
    is.na(x$prevalence),
    NA_character_,
    sprintf("%.2f (%.2f to %.2f)",
            x$prevalence, x$lower, x$upper)
  )
}

get_row <- function(characteristics, design,
                    subgroup_var = NULL, subgroup_level = NULL) {

  x <- weighted_prev(
    design,
    subgroup_var = subgroup_var,
    subgroup_level = subgroup_level
  )

  x <- x[match(TIME_LEVELS, x$TIME), ]

  data.frame(
    Characteristics = characteristics,
    `<2h` = format_prev(x[1, , drop = FALSE]),
    `2h–4h` = format_prev(x[2, , drop = FALSE]),
    `4h–6h` = format_prev(x[3, , drop = FALSE]),
    `>6h` = format_prev(x[4, , drop = FALSE]),
    check.names = FALSE,
    stringsAsFactors = FALSE
  )
}

Table2_parts <- list()

# Overall
Table2_parts[[1]] <- get_row("Total", TABLE2_DESIGN)

# Subgroups: same order as manuscript Table 2
k <- 2

for (v in names(TABLE2_VARS)) {

  Table2_parts[[k]] <- data.frame(
    Characteristics = TABLE2_VARS[[v]],
    `<2h` = "",
    `2h–4h` = "",
    `4h–6h` = "",
    `>6h` = "",
    check.names = FALSE,
    stringsAsFactors = FALSE
  )
  k <- k + 1

  for (lev in TABLE2_LEVELS[[v]]) {

    Table2_parts[[k]] <- get_row(
      paste0("  ", lev),
      TABLE2_DESIGN,
      subgroup_var = v,
      subgroup_level = lev
    )
    k <- k + 1
  }
}

Table2 <- bind_rows(Table2_parts)

Table2


Characteristics,<2h,2h–4h,4h–6h,>6h
<chr>,<chr>,<chr>,<chr>,<chr>
Total,8.71 (7.95 to 9.46),10.45 (10.02 to 10.88),11.49 (11.07 to 11.91),13.90 (13.42 to 14.39)
Sex,,,,
Male,11.41 (10.41 to 12.42),13.39 (12.79 to 13.98),14.62 (13.96 to 15.28),17.63 (16.85 to 18.40)
Female,4.19 (3.36 to 5.01),6.68 (6.21 to 7.15),8.50 (8.02 to 8.98),10.97 (10.42 to 11.53)
Grade,,,,
7th–9th grade (middle school),6.10 (5.30 to 6.89),7.72 (7.23 to 8.21),8.57 (8.03 to 9.10),11.20 (10.53 to 11.87)
10th–12th grade (high school),12.03 (10.71 to 13.35),12.72 (12.09 to 13.36),13.26 (12.69 to 13.83),15.34 (14.70 to 15.98)
Region of residence,,,,
Urban,8.66 (7.89 to 9.44),10.34 (9.90 to 10.79),11.35 (10.92 to 11.79),13.59 (13.09 to 14.09)


## Cell 20 — Export Table 2 to Excel


In [23]:
TABLE2_XLSX <- file.path(DATA_DIR, "Table2_screen_time.xlsx")

wb2 <- createWorkbook()
addWorksheet(wb2, "Table 2")

writeData(
  wb2, "Table 2",
  "Table 2. Prevalence of obesity stratified by hours of daily screen time.",
  startRow = 1, startCol = 1
)

writeData(
  wb2, "Table 2",
  Table2,
  startRow = 3, startCol = 1, rowNames = FALSE
)

writeData(
  wb2, "Table 2",
  "Abbreviations: CI, confidence interval.",
  startRow = nrow(Table2) + 5, startCol = 1
)

header_style2 <- createStyle(
  textDecoration = "bold",
  halign = "center",
  border = "Bottom"
)

section_style2 <- createStyle(
  textDecoration = "bold"
)

addStyle(
  wb2, "Table 2",
  header_style2,
  rows = 3,
  cols = 1:ncol(Table2),
  gridExpand = TRUE
)

section_rows2 <- which(
  Table2$`<2h` == "" &
    Table2$Characteristics != "Total"
) + 3

if (length(section_rows2) > 0) {
  addStyle(
    wb2, "Table 2",
    section_style2,
    rows = section_rows2,
    cols = 1,
    gridExpand = FALSE
  )
}

setColWidths(wb2, "Table 2", cols = 1, widths = 42)
setColWidths(wb2, "Table 2", cols = 2:ncol(Table2), widths = 24)

freezePane(wb2, "Table 2", firstActiveRow = 4)

saveWorkbook(wb2, TABLE2_XLSX, overwrite = TRUE)

cat("Saved:", TABLE2_XLSX, "\n")


Saved: C:/Users/picks/OneDrive/문서/Main/01_Others/논문/비만/Data/Table2_screen_time.xlsx 
